In [ ]:
'''
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()
'''

In [ ]:
'''
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

sovannratana_food_101_path = kagglehub.dataset_download('sovannratana/food-101')

print('Data source import complete.')
'''

In [ ]:
import os
import numpy as np
import torchvision.models as models
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torchvision.transforms import InterpolationMode
import time # Import time to track each model's training time
import matplotlib.pyplot as plt
from torchvision.datasets import Food101


## 2. Load Data

In [ ]:
# ✅ Updated path to the dataset images folder on Kaggle
dataset_path = "/kaggle/input/food-101/food-101/images"

In [ ]:
transform = transforms.Compose([
    transforms.Resize(232, interpolation=InterpolationMode.BILINEAR), # ResNet preprocess use 232 resize
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### [2.1] Data Splitting

In [ ]:
# Updated paths for the Kaggle environment
train_dataset = Food101(root='/kaggle/input/food-101', split='train', transform=transform, download=False)
test_dataset = Food101(root='/kaggle/input/food-101', split='test', transform=transform, download=False)

In [ ]:
from torch.utils.data import random_split
# Define split lengths — e.g., 80% train and 20% validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

# Split the dataset
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])


In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=50, shuffle=True)  # Shuffle training data
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=50, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=50, shuffle=False)

## 3. Load Model

### [3.1] Experiment 1: Unfreeze layer 4

* Layer 4 will be unfreeze first because it is the deepest convolution block that captures high-level abstractions specific to the task, so tuning it should have the highest impact on prediction quality.

In [ ]:
# Load pre-trained ResNet50 model
resnet50 = models.resnet50(pretrained=True)

In [ ]:
for name, param in resnet50.named_parameters():
    if "layer4" in name:
        param.requires_grad = True  # unfreeze only layer4
    else:
        param.requires_grad = False

In [ ]:
in_features = resnet50.fc.in_features

# Replace model head with your own 3-layer MLP
resnet50.fc = nn.Sequential(
    nn.Linear(in_features, 256), # - A Linear layer reducing from 2038 features to 256
    nn.ReLU(),                   # - A ReLU activation function
    nn.Linear(256, 128),         # - A Linear layer reducing from 256 features to 128
    nn.ReLU(),                   # - A ReLU activation functi
    nn.Linear(128, 101),         # - A final Linear layer reducing from 128 to 101 (for 101 class)
)

In [ ]:
# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define loss funtion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet50.parameters(), lr=0.00001)

In [ ]:
# Move model to device
resnet50.to(device)

#### [3.1.2] Initiate Training

In [ ]:
# Initialize lists to store epoch-wise values
train_losses = []       # List to store training losses
train_accuracies = []   # List to store training accuracies
val_losses = []         # List to store validation losses
val_accuracies = []     # List to store validation accuracies

# Start timing
start_time = time.time()

# Training loop
num_epochs = 10                     # Number of epochs for training
for epoch in range(num_epochs):

    # Training
    resnet50.train()             # Set the model to training mode
    running_train_loss = 0.0        # Accumulate training loss
    correct_train = 0               # Count correct predictions in training
    total_train = 0                 # Total training samples processed

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
        optimizer.zero_grad()                                   # Reset gradients to zero
        outputs = resnet50(inputs)                           # Forward pass

        # Unpack logits if model returns tuple
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = criterion(logits, labels)                        # Calculate loss
        loss.backward()                                         # Backward pass
        optimizer.step()                                        # Update model parameters

        running_train_loss += loss.item() * inputs.size(0)      # Update accumulated loss
        _, predicted = torch.max(logits, 1)                     # Get predicted class indices
        total_train += labels.size(0)                           # Update total count
        correct_train += (predicted == labels).sum().item()     # Count correct predictions

    # Calculate epoch-wise training loss and accuracy
    epoch_train_loss = running_train_loss / len(train_loader.dataset)     # Average training loss
    train_accuracy = correct_train / total_train                          # Training accuracy

    # Validation
    resnet50.eval()              # Set the model to evaluation mode
    running_val_loss = 0.0        # Accumulate validation loss
    correct_val = 0               # Count correct predictions in validation
    total_val = 0                 # Total validation samples processed

    with torch.no_grad():  # No need to calculate gradients during validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
            outputs = resnet50(inputs)                           # Forward pass

            # Unpack logits if model returns tuple
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs

            loss = criterion(logits, labels)                        # Calculate loss
            running_val_loss += loss.item() * inputs.size(0)        # Update accumulated loss
            _, predicted = torch.max(logits, 1)                     # Get predicted class indices
            total_val += labels.size(0)                             # Update total count
            correct_val += (predicted == labels).sum().item()       # Count correct predictions

    # Calculate epoch-wise validation loss and accuracy
    epoch_val_loss = running_val_loss / len(val_loader.dataset)       # Average validation loss
    val_accuracy = correct_val / total_val                            # Validation accuracy

    # Print progress for the current epoch.
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

    # Append values to the lists
    train_losses.append(epoch_train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(val_accuracy)

# End timing
end_time = time.time()
elapsed_time = end_time - start_time

print(f'\nTotal Training Time: {elapsed_time // 60:.0f} minutes and {elapsed_time % 60:.2f} seconds')


In [ ]:
# Plot training and validation losses starting from index 1
epochs = range(1, len(train_losses) + 1)  # Generate the range of epochs starting from 1

# Plot training and validation accuracies
plt.plot(epochs, train_accuracies, label='Training Accuracy')   # Plot training accuracies over epochs
plt.plot(epochs, val_accuracies, label='Validation Accuracy')   # Plot validation accuracies over epochs
plt.xlabel('Epoch')                                             # Set label for the x-axis
plt.ylabel('Accuracy')                                          # Set label for the y-axis
plt.title('Training and Validation Accuracies')                 # Set title for the plot
plt.legend()                                                    # Display legend
plt.show()

In [ ]:
# Plot training and validation losses starting from index 1
epochs = range(1, len(train_losses) + 1)  # Generate the range of epochs starting from 1

# Plot training and validation losses
plt.plot(epochs, train_losses, label='Training Loss')   # Plot training losses over epochs
plt.plot(epochs, val_losses, label='Validation Loss')   # Plot validation losses over epochs
plt.xlabel('Epoch')                                     # Set label for the x-axis
plt.ylabel('Loss')                                      # Set label for the y-axis
plt.title('Training and Validation Losses')             # Set title for the plot
plt.legend()                                            # Display legend
plt.grid(True)                                          # Display grid
plt.show()                                              # Show the plot

* After unfreezing layer 4 we see an increase in accuracies with smaller epochs. The optimal range is around 4 Epochs with around 64% accuracies. The model starts to overfit on training set after that.
* In future, experiment we will try to reduce this overfitting and also unfreezing layer 3 to see if model improves further.

### [3.2] Experiment 3: Unfreeze layer 3 & 4

* Unfreeze layer 3 in addition to layer 4
* Introduce data augmentation: random flip, crop, and rotations to reduce overfitting
* Lower batch size to help reduce overfitting
* Introduce early stoppage scheduler
* Use a discriminative learning rate schedule:
    * Lower LR for earlier layers (1e-5)
    * Higher LR for later layers (1e-4 or 1e-3)

#### [3.2.1] Data Augmentation

1. Random Resize Crop 224
2. Random Horizontal Flip
3. RandomRotation 15

In [ ]:
# Define transformations
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),                                            # Horizontally flip the given image randomly
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize using ImageNet's mean and std values.
])

transform_test = transforms.Compose([
    transforms.Resize(232, interpolation=InterpolationMode.BILINEAR), # ResNet preprocess use 232 resize
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Updated paths for the Kaggle environment
train_dataset = Food101(root='/kaggle/input/food-101', split='train', transform=transform_train, download=False)
test_dataset = Food101(root='/kaggle/input/food-101', split='test', transform=transform_test, download=False)


In [ ]:
dataset_size = len(train_dataset)
val_ratio = 0.2  # 20% validation

train_size = int((1 - val_ratio) * dataset_size)
val_size = dataset_size - train_size  # ensures sum matches exactly

print (train_size)
print (val_size)

In [ ]:
# Split the dataset
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=50, shuffle=True)  # Shuffle training data
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=50, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=50, shuffle=False)

In [ ]:
print(f"Size of train_loader: {len(train_loader)}")
print(f"Size of val_loader: {len(val_loader)}")
print(f"Size of test_loader: {len(test_loader)}")

#### Define custom_callback class to introduce early stoppage

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

class CustomCallback:
    def __init__(self, early_stop_patience=5, reduce_lr_factor=0.2, reduce_lr_patience=3, reduce_lr_min_lr=0.0000001, checkpoint_path='checkpoint.pth', log_dir='logs'):

        # Parameters for early stopping and learning rate reduction
        self.early_stop_patience = early_stop_patience    # Patience for early stopping
        self.reduce_lr_factor = reduce_lr_factor          # Factor by which to reduce learning rate
        self.reduce_lr_patience = reduce_lr_patience      # Patience for reducing learning rate
        self.reduce_lr_min_lr = reduce_lr_min_lr          # Minimum learning rate
        self.checkpoint_path = checkpoint_path            # Path to save model checkpoints
        # self.log_dir = log_dir  # Directory for logging (if needed)

        self.early_stop_counter = 0                       # Counter to track epochs without improvement
        self.best_val_loss = float('inf')                 # Best validation loss

        self.optimizer = None                             # Placeholder for the optimizer
        self.scheduler = None                             # Placeholder for the LR scheduler

    def set_optimizer(self, optimizer):
        self.optimizer = optimizer                        # Set optimizer for training

    def on_epoch_end(self, epoch, val_loss):
        # Early Stopping: Check if the validation loss improved
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.early_stop_counter = 0                   # Reset counter if improvement is seen
        else:
            self.early_stop_counter += 1                  # Increment counter if no improvement

        # Trigger early stopping if no improvement for a defined number of epochs
        if self.early_stop_counter >= self.early_stop_patience:
            print("Early stopping triggered!")
            return True                                   #  Return True to signal stopping training

        # Adjust the learning rate if a plateau in validation loss is detected
        if self.scheduler is not None:
            self.scheduler.step(val_loss)                 # Adjust learning rate based on validation loss

        return False  # Continue training if conditions are not met

    def on_train_begin(self):
        # Initialize the ReduceLROnPlateau scheduler with the optimizer and parameters.
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode='min', factor=self.reduce_lr_factor,patience=self.reduce_lr_patience, min_lr=self.reduce_lr_min_lr)

    def on_train_end(self):
        pass                  # Additional cleanup can be done here if needed

    def set_model(self, model):
        self.model = model    # Store the model reference

#### [3.2.2] Initiate Model Training

In [ ]:
# Load pre-trained ResNet50 model
resnet50 = models.resnet50(pretrained=True)

for name, param in resnet50.named_parameters():
    if "layer4" or "layer3" in name:
        param.requires_grad = True  # unfreeze only layer4
    else:
        param.requires_grad = False

In [ ]:
in_features = resnet50.fc.in_features

# Replace model head with your own 3-layer MLP
resnet50.fc = nn.Sequential(
    nn.Linear(in_features, 256), # - A Linear layer reducing from 1024 features to 256
    nn.ReLU(),                   # - A ReLU activation function
    nn.Linear(256, 128),         # - A Linear layer reducing from 256 features to 128
    nn.ReLU(),                   # - A ReLU activation functi
    nn.Linear(128, 101),         # - A final Linear layer reducing from 128 to 101 (for 101 class)
)

In [ ]:
# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define loss funtion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {'params': resnet50.fc.parameters(), 'lr': 1e-3},
    {'params': resnet50.layer4.parameters(), 'lr': 1e-4},
    {'params': resnet50.layer3.parameters(), 'lr': 1e-5}
])

Lower LR for earlier layers (more general), higher for later layers (more task-specific).

In [ ]:
# Initiate custom_callback
custom_callback = CustomCallback()
custom_callback.set_optimizer(optimizer)

In [ ]:
# Move model to device
resnet50.to(device)

In [ ]:
# Set the model for the custom callback
custom_callback.set_model(resnet50)

In [ ]:
#Create a learning rate scheduler for reducing LR on plateau based on validation loss.
lr_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, min_lr=1e-6)

In [ ]:
best_val_loss = float('inf')  # Initialize best validation loss
best_model_path = '/kaggle/working/best_resnet50.pth'  # Path to save the best model

In [ ]:
# Initialize lists to store epoch-wise values
train_losses = []       # List to store training losses
train_accuracies = []   # List to store training accuracies
val_losses = []         # List to store validation losses
val_accuracies = []     # List to store validation accuracies

# Start timing
start_time = time.time()

# Training loop
num_epochs = 15                     # Number of epochs for training
for epoch in range(num_epochs):

    # Training
    resnet50.train()             # Set the model to training mode
    running_train_loss = 0.0        # Accumulate training loss
    correct_train = 0               # Count correct predictions in training
    total_train = 0                 # Total training samples processed

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
        optimizer.zero_grad()                                   # Reset gradients to zero
        outputs = resnet50(inputs)                           # Forward pass

        # Unpack logits if model returns tuple
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = criterion(logits, labels)                        # Calculate loss
        loss.backward()                                         # Backward pass
        optimizer.step()                                        # Update model parameters

        running_train_loss += loss.item() * inputs.size(0)      # Update accumulated loss
        _, predicted = torch.max(logits, 1)                     # Get predicted class indices
        total_train += labels.size(0)                           # Update total count
        correct_train += (predicted == labels).sum().item()     # Count correct predictions

    # Calculate epoch-wise training loss and accuracy
    epoch_train_loss = running_train_loss / len(train_loader.dataset)     # Average training loss
    train_accuracy = correct_train / total_train                          # Training accuracy

    # Validation
    resnet50.eval()              # Set the model to evaluation mode
    running_val_loss = 0.0        # Accumulate validation loss
    correct_val = 0               # Count correct predictions in validation
    total_val = 0                 # Total validation samples processed

    with torch.no_grad():  # No need to calculate gradients during validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
            outputs = resnet50(inputs)                           # Forward pass

            # Unpack logits if model returns tuple
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs

            loss = criterion(logits, labels)                        # Calculate loss
            running_val_loss += loss.item() * inputs.size(0)        # Update accumulated loss
            _, predicted = torch.max(logits, 1)                     # Get predicted class indices
            total_val += labels.size(0)                             # Update total count
            correct_val += (predicted == labels).sum().item()       # Count correct predictions

    # Calculate epoch-wise validation loss and accuracy
    epoch_val_loss = running_val_loss / len(val_loader.dataset)       # Average validation loss
    val_accuracy = correct_val / total_val                            # Validation accuracy

    # Print progress for the current epoch.
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

    # Append values to the lists
    train_losses.append(epoch_train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(val_accuracy)

    # Save the model if the validation loss has improved
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(resnet50.state_dict(), best_model_path)
        print(f'✅ Saved Best Model at Epoch {epoch+1} with Val Loss: {epoch_val_loss:.4f}')

    # Update learning rate scheduler based on validation loss.
    lr_scheduler.step(epoch_val_loss)                               # Adjust learning rate based on validation loss

    # Check for early stopping using the custom callback.
    if custom_callback.on_epoch_end(epoch, epoch_val_loss):
        break

# End timing
end_time = time.time()
elapsed_time = end_time - start_time

print(f'\nTotal Training Time: {elapsed_time // 60:.0f} minutes and {elapsed_time % 60:.2f} seconds')


Key learnings:
* Overfiting still persist although there are some improvement. Further data augmentation may be necessary
* Performance peaked at around 71% accuracy after unfreezing the 2 deeper layers.
* To further enhance result we will explore deeper layer of ResNet, Resnet-101 that has double the layer of Resnet 50.

### [3.3] Experiment 4: Resnet 101

In this experiment we will further reduce overfitting and train on deeper Resnet model to boost performance.

#### [3.3.1] Data Augmentation

* Add Color Jitter in Data Augmentation to improve model generalizability
* Introduce label_smoothing to optimizer


In [ ]:
# Define transformations
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),                                            # Horizontally flip the given image randomly
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize using ImageNet's mean and std values.
])

transform_test = transforms.Compose([
    transforms.Resize(232, interpolation=InterpolationMode.BILINEAR), # ResNet preprocess use 232 resize
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Updated paths for the Kaggle environment
train_dataset = Food101(root=sovannratana_food_101_path, split='train', transform=transform_train, download=False)
test_dataset = Food101(root=sovannratana_food_101_path, split='test', transform=transform_test, download=False)

In [ ]:
dataset_size = len(train_dataset)
val_ratio = 0.2  # 20% validation

train_size = int((1 - val_ratio) * dataset_size)
val_size = dataset_size - train_size  # ensures sum matches exactly

print (train_size)
print (val_size)

In [ ]:
# Split the dataset
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)  # Shuffle training data
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

#### [3.3.2] Train Model

In [ ]:
# Load pre-trained ResNet101 model
resnet101 = models.resnet101(pretrained=True)

In [ ]:
# Load pre-trained ResNet50 model
for name, param in resnet101.named_parameters():
    if "layer4" or "layer3" in name:
        param.requires_grad = True  # unfreeze only layer4
    else:
        param.requires_grad = False

In [ ]:
in_features = resnet101.fc.in_features

In [ ]:
# Replace model head with your own 3-layer MLP
resnet101.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 101)
)

In [ ]:
# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define loss funtion and optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam([
    {'params': resnet101.fc.parameters(), 'lr': 1e-3},
    {'params': resnet101.layer4.parameters(), 'lr': 1e-4},
    {'params': resnet101.layer3.parameters(), 'lr': 1e-5}
])

In [ ]:
# Initiate custom_callback
custom_callback = CustomCallback()
custom_callback.set_optimizer(optimizer)

In [ ]:
# Move model to device
resnet101.to(device)

In [ ]:
# Set the model for the custom callback
custom_callback.set_model(resnet101)

In [ ]:
#Create a learning rate scheduler for reducing LR on plateau based on validation loss.
lr_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, min_lr=1e-6)

In [ ]:
# Initialize lists to store epoch-wise values
train_losses = []       # List to store training losses
train_accuracies = []   # List to store training accuracies
val_losses = []         # List to store validation losses
val_accuracies = []     # List to store validation accuracies

# Start timing
start_time = time.time()

# Training loop
num_epochs = 7                     # Number of epochs for training
for epoch in range(num_epochs):

    # Training
    resnet101.train()             # Set the model to training mode
    running_train_loss = 0.0        # Accumulate training loss
    correct_train = 0               # Count correct predictions in training
    total_train = 0                 # Total training samples processed

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
        optimizer.zero_grad()                                   # Reset gradients to zero
        outputs = resnet101(inputs)                           # Forward pass

        # Unpack logits if model returns tuple
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = criterion(logits, labels)                        # Calculate loss
        loss.backward()                                         # Backward pass
        optimizer.step()                                        # Update model parameters

        running_train_loss += loss.item() * inputs.size(0)      # Update accumulated loss
        _, predicted = torch.max(logits, 1)                     # Get predicted class indices
        total_train += labels.size(0)                           # Update total count
        correct_train += (predicted == labels).sum().item()     # Count correct predictions

    # Calculate epoch-wise training loss and accuracy
    epoch_train_loss = running_train_loss / len(train_loader.dataset)     # Average training loss
    train_accuracy = correct_train / total_train                          # Training accuracy

    # Validation
    resnet101.eval()              # Set the model to evaluation mode
    running_val_loss = 0.0        # Accumulate validation loss
    correct_val = 0               # Count correct predictions in validation
    total_val = 0                 # Total validation samples processed

    with torch.no_grad():  # No need to calculate gradients during validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
            outputs = resnet101(inputs)                           # Forward pass

            # Unpack logits if model returns tuple
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs

            loss = criterion(logits, labels)                        # Calculate loss
            running_val_loss += loss.item() * inputs.size(0)        # Update accumulated loss
            _, predicted = torch.max(logits, 1)                     # Get predicted class indices
            total_val += labels.size(0)                             # Update total count
            correct_val += (predicted == labels).sum().item()       # Count correct predictions

    # Calculate epoch-wise validation loss and accuracy
    epoch_val_loss = running_val_loss / len(val_loader.dataset)       # Average validation loss
    val_accuracy = correct_val / total_val                            # Validation accuracy

    # Print progress for the current epoch.
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

    # Append values to the lists
    train_losses.append(epoch_train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(val_accuracy)

    # Update learning rate scheduler based on validation loss.
    lr_scheduler.step(epoch_val_loss)                               # Adjust learning rate based on validation loss

    # Check for early stopping using the custom callback.
    if custom_callback.on_epoch_end(epoch, epoch_val_loss):
        break

# End timing
end_time = time.time()
elapsed_time = end_time - start_time

print(f'\nTotal Training Time: {elapsed_time // 60:.0f} minutes and {elapsed_time % 60:.2f} seconds')


In [ ]:
# Plot training and validation losses starting from index 1
epochs = range(1, len(train_losses) + 1)  # Generate the range of epochs starting from 1

# Plot training and validation accuracies
plt.plot(epochs, train_accuracies, label='Training Accuracy')   # Plot training accuracies over epochs
plt.plot(epochs, val_accuracies, label='Validation Accuracy')   # Plot validation accuracies over epochs
plt.xlabel('Epoch')                                             # Set label for the x-axis
plt.ylabel('Accuracy')                                          # Set label for the y-axis
plt.title('Training and Validation Accuracies')                 # Set title for the plot
plt.legend()                                                    # Display legend
plt.show()

In [ ]:
# Plot training and validation losses starting from index 1
epochs = range(1, len(train_losses) + 1)  # Generate the range of epochs starting from 1

# Plot training and validation losses
plt.plot(epochs, train_losses, label='Training Loss')   # Plot training losses over epochs
plt.plot(epochs, val_losses, label='Validation Loss')   # Plot validation losses over epochs
plt.xlabel('Epoch')                                     # Set label for the x-axis
plt.ylabel('Loss')                                      # Set label for the y-axis
plt.title('Training and Validation Losses')             # Set title for the plot
plt.legend()                                            # Display legend
plt.grid(True)                                          # Display grid
plt.show()                                              # Show the plot

* Terminate early due to overfitting observed
* Resnet-101 performs just marginally better than ResNet50 (~0.66 best resut, at 7 epoch). Overall, I think the additional training time does not justify this marginal increase in result.
* Future experiment will use Resnet50, further finetuned for less overfitting.

### [3.4] Experiment 5: ResNet 50 (unfreeze layer 2, 3 & 4, more advanced data augmentation and modify model head)

* In an attempt to further reduce overfitting we will modify the model head to include dropouts as well as batchnorm to boost convergence.
* Reduce batch size to 32

#### [3.4.1] Data Augmentation

In [ ]:
# Define transformations
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(150),
    transforms.RandomHorizontalFlip(),                                            # Horizontally flip the given image randomly
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize using ImageNet's mean and std values.
])

transform_test = transforms.Compose([
    transforms.Resize(232, interpolation=InterpolationMode.BILINEAR), # ResNet preprocess use 232 resize
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Updated paths for the Kaggle environment
train_dataset = Food101(root='/kaggle/input/food-101', split='train', transform=transform_train, download=False)
test_dataset = Food101(root='/kaggle/input/food-101', split='test', transform=transform_test, download=False)

dataset_size = len(train_dataset)
val_ratio = 0.2  # 20% validation

train_size = int((1 - val_ratio) * dataset_size)
val_size = dataset_size - train_size  # ensures sum matches exactly

print (train_size)
print (val_size)


In [ ]:
# Split the dataset
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)  # Shuffle training data
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

#### [3.4.2] Train Model

In [ ]:
# Load pre-trained ResNet101 model
resnet50 = models.resnet50(pretrained=True)

In [ ]:
# Load pre-trained ResNet50 model
for name, param in resnet50.named_parameters():
if any(layer in name for layer in ["layer2", "layer3", "layer4"]):
    param.requires_grad = True
else:
    param.requires_grad = False

In [ ]:
in_features = resnet50.fc.in_features

In [ ]:
# Replace model head with your own 3-layer MLP
resnet50.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 101)
)

In [ ]:
# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define loss funtion and optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam([
    {'params': resnet50.fc.parameters(), 'lr': 1e-3},
    {'params': resnet50.layer4.parameters(), 'lr': 1e-4},
    {'params': resnet50.layer3.parameters(), 'lr': 1e-5},
    {'params': resnet50.layer2.parameters(), 'lr': 1e-5}
])

# Initiate custom_callback
custom_callback = CustomCallback()
custom_callback.set_optimizer(optimizer)

# Move model to device
resnet50.to(device)

In [ ]:
# Set the model for the custom callback
custom_callback.set_model(resnet50)

In [ ]:
#Create a learning rate scheduler for reducing LR on plateau based on validation loss.
lr_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, min_lr=1e-6)

In [ ]:
# Initialize lists to store epoch-wise values
train_losses = []       # List to store training losses
train_accuracies = []   # List to store training accuracies
val_losses = []         # List to store validation losses
val_accuracies = []     # List to store validation accuracies

# Start timing
start_time = time.time()

# Training loop
num_epochs = 13                     # Number of epochs for training
for epoch in range(num_epochs):

    # Training
    resnet50.train()             # Set the model to training mode
    running_train_loss = 0.0        # Accumulate training loss
    correct_train = 0               # Count correct predictions in training
    total_train = 0                 # Total training samples processed

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
        optimizer.zero_grad()                                   # Reset gradients to zero
        outputs = resnet50(inputs)                           # Forward pass

        # Unpack logits if model returns tuple
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = criterion(logits, labels)                        # Calculate loss
        loss.backward()                                         # Backward pass
        optimizer.step()                                        # Update model parameters

        running_train_loss += loss.item() * inputs.size(0)      # Update accumulated loss
        _, predicted = torch.max(logits, 1)                     # Get predicted class indices
        total_train += labels.size(0)                           # Update total count
        correct_train += (predicted == labels).sum().item()     # Count correct predictions

    # Calculate epoch-wise training loss and accuracy
    epoch_train_loss = running_train_loss / len(train_loader.dataset)     # Average training loss
    train_accuracy = correct_train / total_train                          # Training accuracy

    # Validation
    resnet50.eval()              # Set the model to evaluation mode
    running_val_loss = 0.0        # Accumulate validation loss
    correct_val = 0               # Count correct predictions in validation
    total_val = 0                 # Total validation samples processed

    with torch.no_grad():  # No need to calculate gradients during validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU if available
            outputs = resnet50(inputs)                           # Forward pass

            # Unpack logits if model returns tuple
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs

            loss = criterion(logits, labels)                        # Calculate loss
            running_val_loss += loss.item() * inputs.size(0)        # Update accumulated loss
            _, predicted = torch.max(logits, 1)                     # Get predicted class indices
            total_val += labels.size(0)                             # Update total count
            correct_val += (predicted == labels).sum().item()       # Count correct predictions

    # Calculate epoch-wise validation loss and accuracy
    epoch_val_loss = running_val_loss / len(val_loader.dataset)       # Average validation loss
    val_accuracy = correct_val / total_val                            # Validation accuracy

    # Print progress for the current epoch.
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

    # Append values to the lists
    train_losses.append(epoch_train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(epoch_val_loss)
    val_accuracies.append(val_accuracy)

    # Save the model if the validation loss has improved
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(resnet50.state_dict(), best_model_path)
        print(f'✅ Saved Best Model at Epoch {epoch+1} with Val Loss: {epoch_val_loss:.4f}')

    # Update learning rate scheduler based on validation loss.
    lr_scheduler.step(epoch_val_loss)                               # Adjust learning rate based on validation loss

    # Check for early stopping using the custom callback.
    if custom_callback.on_epoch_end(epoch, epoch_val_loss):
        break

# End timing
end_time = time.time()
elapsed_time = end_time - start_time

print(f'\nTotal Training Time: {elapsed_time // 60:.0f} minutes and {elapsed_time % 60:.2f} seconds')


In [ ]:
# Plot training and validation losses starting from index 1
epochs = range(1, len(train_losses) + 1)  # Generate the range of epochs starting from 1

# Plot training and validation accuracies
plt.plot(epochs, train_accuracies, label='Training Accuracy')   # Plot training accuracies over epochs
plt.plot(epochs, val_accuracies, label='Validation Accuracy')   # Plot validation accuracies over epochs
plt.xlabel('Epoch')                                             # Set label for the x-axis
plt.ylabel('Accuracy')                                          # Set label for the y-axis
plt.title('Training and Validation Accuracies')                 # Set title for the plot
plt.legend()                                                    # Display legend
plt.show()

In [ ]:
# Plot training and validation losses starting from index 1
epochs = range(1, len(train_losses) + 1)  # Generate the range of epochs starting from 1

# Plot training and validation losses
plt.plot(epochs, train_losses, label='Training Loss')   # Plot training losses over epochs
plt.plot(epochs, val_losses, label='Validation Loss')   # Plot validation losses over epochs
plt.xlabel('Epoch')                                     # Set label for the x-axis
plt.ylabel('Loss')                                      # Set label for the y-axis
plt.title('Training and Validation Losses')             # Set title for the plot
plt.legend()                                            # Display legend
plt.grid(True)                                          # Display grid
plt.show()                                              # Show the plot

#### Evaluate on testingset

In [ ]:
# Evaluate on Testing set
# Set the model to evaluation mode and disable gradients for testing.
resnet50.eval()
test_correct = 0          # Counter for correct predictions in testing
test_total = 0            # Total test samples processed
test_running_loss = 0.0   # Accumulator for test loss

with torch.no_grad():                                           # Turn off gradients during evaluation
    for inputs, labels in test_loader:                          # Iterate through test data
        inputs, labels = inputs.to(device), labels.to(device)   # Move data to GPU
        outputs = resnet50(inputs)                           # Get model predictions

        if isinstance(outputs, tuple):
            logits = outputs[0]                                 # Unpack logits from model outputs if necessary
        else:
            logits = outputs
        loss = criterion(logits, labels)                        # Calculate loss

        test_running_loss += loss.item() * inputs.size(0)       # Update running test loss
        _, predicted = torch.max(logits, 1)                     # Get predicted labels
        test_total += labels.size(0)                            # Update total number of samples
        test_correct += (predicted == labels).sum().item()      # Update number of correctly predicted samples

# Calculate test loss and accuracy
test_loss = test_running_loss / len(test_loader.dataset)  # Average test loss
test_accuracy = test_correct / test_total                 # Test accuracy

# Print test results
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}')